In [8]:
from keras.utils import to_categorical
from keras.preprocessing.image import load_img
from keras.models import Sequential
from keras.layers import Dense, Conv2D, Dropout, Flatten, MaxPooling2D

import os
import pandas as pd
import numpy as np

In [9]:
TRAIN_DIR= 'images/train'
TEST_DIR= 'images/test'


In [10]:
def createdataframe(dir):
    image_paths = []
    labels = []

    for label in os.listdir(dir):
        for imagename in os.listdir(os.path.join(dir, label)):
            image_paths.append(os.path.join(dir, label, imagename))
            labels.append(label)

        print(label, "completed")

    return image_paths, labels

In [11]:
train = pd.DataFrame()
train['image'], train['label'] = createdataframe(TRAIN_DIR)

angry completed
disgust completed
fear completed
happy completed
neutral completed
sad completed
surprise completed


In [12]:
print (train)


                                image     label
0            images/train/angry/0.jpg     angry
1            images/train/angry/1.jpg     angry
2           images/train/angry/10.jpg     angry
3        images/train/angry/10002.jpg     angry
4        images/train/angry/10016.jpg     angry
...                               ...       ...
28816  images/train/surprise/9969.jpg  surprise
28817  images/train/surprise/9985.jpg  surprise
28818  images/train/surprise/9990.jpg  surprise
28819  images/train/surprise/9992.jpg  surprise
28820  images/train/surprise/9996.jpg  surprise

[28821 rows x 2 columns]


In [13]:
test = pd.DataFrame()
test['image'], test['label'] = createdataframe(TEST_DIR)

angry completed
disgust completed
fear completed
happy completed
neutral completed
sad completed
surprise completed


In [14]:
print(test)
print(test['image'])

                              image     label
0       images/test/angry/10052.jpg     angry
1       images/test/angry/10065.jpg     angry
2       images/test/angry/10079.jpg     angry
3       images/test/angry/10095.jpg     angry
4       images/test/angry/10121.jpg     angry
...                             ...       ...
7061  images/test/surprise/9806.jpg  surprise
7062  images/test/surprise/9830.jpg  surprise
7063  images/test/surprise/9853.jpg  surprise
7064  images/test/surprise/9878.jpg  surprise
7065   images/test/surprise/993.jpg  surprise

[7066 rows x 2 columns]
0         images/test/angry/10052.jpg
1         images/test/angry/10065.jpg
2         images/test/angry/10079.jpg
3         images/test/angry/10095.jpg
4         images/test/angry/10121.jpg
                    ...              
7061    images/test/surprise/9806.jpg
7062    images/test/surprise/9830.jpg
7063    images/test/surprise/9853.jpg
7064    images/test/surprise/9878.jpg
7065     images/test/surprise/993.jpg
Name:

In [15]:
from tqdm.notebook import tqdm

In [16]:
def extract_features(images):
    features = []

    for image in tqdm(images):
        img = load_img(image, color_mode="grayscale")
        img = np.array(img)
        features.append(img)

    features = np.array(features)
    features = features.reshape(len(features), 48, 48, 1)

    return features

In [35]:
train_features = extract_features(train['image'])

  0%|          | 0/28821 [00:00<?, ?it/s]

In [36]:
%pip install --upgrade --force-reinstall Pillow

  Using cached pillow-12.3.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (9.1 kB)
Using cached pillow-12.3.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (6.9 MB)
  Attempting uninstall: Pillow
    Found existing installation: pillow 12.3.0
    Uninstalling pillow-12.3.0:
      Successfully uninstalled pillow-12.3.0
Note: you may need to restart the kernel to use updated packages.


In [37]:
from PIL import Image

print("Pillow is working!")

Pillow is working!


In [38]:
test_features = extract_features(test['image'])

  0%|          | 0/7066 [00:00<?, ?it/s]

In [39]:
x_train = train_features/255.0
x_test = test_features/255.0

In [40]:
%pip install -U scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [41]:
import sklearn

print(sklearn.__version__)

1.9.1


In [42]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [43]:
le = LabelEncoder()
le.fit(train['label'])

Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)Holds the label for each class.","ndarray[object](7,)","['angry','disgust','fear',...,'neutral','sad','surprise']"


In [44]:
y_train = le.transform(train['label'])
y_test = le.transform(test['label'])

In [45]:
y_train = to_categorical(y_train,num_classes = 7)
y_test = to_categorical(y_test,num_classes = 7)

In [46]:
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Dropout, Flatten, Dense

In [47]:
output_class = 7

In [48]:
from keras.models import Sequential
from keras.layers import (
    Conv2D, MaxPooling2D, Dropout, Flatten, Dense
)

model = Sequential([
    Conv2D(128, (3, 3), activation='relu',
           input_shape=(48, 48, 1)),
    MaxPooling2D((2, 2)),
    Dropout(0.4),

    Conv2D(256, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Dropout(0.4),

    Conv2D(512, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Dropout(0.4),

    Conv2D(512, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Dropout(0.4),

    Flatten(),
    Dense(512, activation='relu'),
    Dropout(0.4),

    Dense(256, activation='relu'),
    Dropout(0.3),

    Dense(7, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Model rebuilt and compiled successfully!")

Model rebuilt and compiled successfully!


/home/rkms1/face_emotion_detection_gpu/.venv-gpu/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [49]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [50]:
print("x_train:", x_train.shape)
print("y_train:", y_train.shape)
print("x_test:", x_test.shape)
print("y_test:", y_test.shape)
print("Model output:", model.output_shape)

x_train: (28821, 48, 48, 1)
y_train: (28821, 7)
x_test: (7066, 48, 48, 1)
y_test: (7066, 7)
Model output: (None, 7)


In [51]:
model.fit(
    x=x_train,
    y=y_train,
    batch_size=128,
    epochs=100,
    validation_data=(x_test, y_test)
)

Epoch 1/100
  5/226 ━━━━━━━━━━━━━━━━━━━━ 3:16 891ms/step - accuracy: 0.1969 - loss: 1.8982

KeyboardInterrupt: 

In [52]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

TensorFlow version: 2.21.0
GPU devices: []


In [30]:
print("Training images:", x_train.shape)
print("Training labels:", y_train.shape)
print("Validation images:", x_test.shape)
print("Validation labels:", y_test.shape)
print("Model output:", model.output_shape)

Training images: (28821, 48, 48, 1)
Training labels: (28821, 7)
Validation images: (7066, 48, 48, 1)
Validation labels: (7066, 7)
Model output: (None, 7)


In [53]:
model_json = model.to_json()

with open("emotiondetector.json", "w") as json_file:
    json_file.write(model_json)

model.save("emotiondetector.keras")

In [ ]:
from tensorflow.keras.models import model_from_json, load_model